# 00 — Rebuild Sequential Weekly Team State

This notebook rebuilds the 2026 in-season team-strength chain using one consistent sequential method:

**Frozen preseason → Week 1 evidence → corrected Week 2 state → Week 2 evidence → corrected Week 3 state**

It does **not** overwrite the frozen preseason model or the original Week 1/Week 2 projection artifacts.

### Architecture

This version deliberately separates:

- **Persistent state**: one row per team containing only information that should carry into the next week.
- **Weekly evidence**: one row per team/game containing that week's actual-vs-expected performance.

That separation prevents old update columns from colliding with new update columns when a new weekly state is created.


## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## 2. Paths and provisional update parameters

In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026

PRESEASON_STRENGTH_PATH = PROCESSED_DIR / "2026_team_strength.parquet"
PRESEASON_PREDICTIONS_PATH = PROCESSED_DIR / "2026_game_predictions.parquet"
CURRENT_SCHEDULE_PATH = WEEKLY_DATA_DIR / "2026_schedule_current.parquet"

WEEK1_EVIDENCE_OUTPUT_PATH = WEEKLY_DATA_DIR / "week_01_sequential_update_features.parquet"
WEEK2_EVIDENCE_OUTPUT_PATH = WEEKLY_DATA_DIR / "week_02_sequential_update_features.parquet"

WEEK2_STATE_OUTPUT_PATH = WEEKLY_DATA_DIR / "week_02_sequential_team_strength.parquet"
WEEK3_STATE_OUTPUT_PATH = WEEKLY_DATA_DIR / "week_03_sequential_team_strength.parquet"

# Provisional. These should eventually be selected through historical backtesting.
RESIDUAL_SCALE = 14.0
MAX_WEEKLY_UPDATE = 1.5

# Separate scoring diagnostics; not used to determine overall strength update.
OPPONENT_CONTEXT_RATE = 0.25
OFFENSE_DEFENSE_UPDATE_RATE = 0.15

print("Preseason strength:", PRESEASON_STRENGTH_PATH)
print("Frozen predictions:", PRESEASON_PREDICTIONS_PATH)
print("Current schedule:", CURRENT_SCHEDULE_PATH)


Preseason strength: ..\..\data\processed\2026_team_strength.parquet
Frozen predictions: ..\..\data\processed\2026_game_predictions.parquet
Current schedule: ..\..\data\processed\weekly\2026_schedule_current.parquet


## 3. Load frozen inputs and current results

In [3]:
preseason_raw = pd.read_parquet(PRESEASON_STRENGTH_PATH).copy()
preseason_predictions = pd.read_parquet(PRESEASON_PREDICTIONS_PATH).copy()
schedule = pd.read_parquet(CURRENT_SCHEDULE_PATH).copy()

required_strength = [
    "team",
    "strength_rank",
    "team_strength",
    "baseline_team_strength",
    "personnel_adjustment",
    "personnel_strength",
    "roster_continuity",
    "roster_continuity_adjustment"
]

required_predictions = [
    "game_id",
    "week",
    "away_team",
    "home_team",
    "home_field_adjustment",
    "rest_adjustment",
    "expected_home_margin"
]

required_schedule = [
    "game_id",
    "week",
    "away_team",
    "home_team",
    "away_score",
    "home_score"
]

for label, df, required in [
    ("preseason strength", preseason_raw, required_strength),
    ("preseason predictions", preseason_predictions, required_predictions),
    ("schedule", schedule, required_schedule),
]:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing {label} columns: {missing}")

print("Preseason teams:", preseason_raw["team"].nunique())
print("Frozen prediction rows:", len(preseason_predictions))
print("Schedule rows:", len(schedule))


Preseason teams: 32
Frozen prediction rows: 272
Schedule rows: 272


## 4. Helper functions

The important design rule is that `build_next_state()` selects only persistent columns from the prior state before merging the new week's evidence. Old weekly evidence can therefore never collide with new weekly evidence.


In [4]:
def make_preseason_state(preseason_df):
    state = preseason_df[
        [
            "team",
            "strength_rank",
            "team_strength",
            "baseline_team_strength",
            "personnel_adjustment",
            "personnel_strength",
            "roster_continuity",
            "roster_continuity_adjustment"
        ]
    ].rename(
        columns={
            "strength_rank": "preseason_strength_rank",
            "team_strength": "preseason_team_strength",
            "baseline_team_strength": "preseason_baseline_team_strength",
            "personnel_adjustment": "preseason_personnel_adjustment",
            "personnel_strength": "preseason_personnel_strength",
            "roster_continuity": "preseason_roster_continuity",
            "roster_continuity_adjustment": "preseason_roster_continuity_adjustment"
        }
    ).copy()

    state["state_week"] = 1
    state["weekly_strength_rank"] = state["preseason_strength_rank"]
    state["weekly_team_strength"] = state["preseason_team_strength"]
    state["cumulative_offense_adjustment"] = 0.0
    state["cumulative_defense_adjustment"] = 0.0

    return state


def games_to_team_rows(games, expected_home_margin_col):
    home = pd.DataFrame({
        "game_id": games["game_id"].values,
        "week": games["week"].values,
        "team": games["home_team"].values,
        "opponent": games["away_team"].values,
        "is_home": 1,
        "points_for": games["home_score"].values,
        "points_against": games["away_score"].values,
        "expected_margin": games[expected_home_margin_col].values
    })

    away = pd.DataFrame({
        "game_id": games["game_id"].values,
        "week": games["week"].values,
        "team": games["away_team"].values,
        "opponent": games["home_team"].values,
        "is_home": 0,
        "points_for": games["away_score"].values,
        "points_against": games["home_score"].values,
        "expected_margin": -games[expected_home_margin_col].values
    })

    out = pd.concat([home, away], ignore_index=True)

    out["point_margin"] = out["points_for"] - out["points_against"]
    out["performance_residual"] = out["point_margin"] - out["expected_margin"]

    return out


def attach_entering_state(team_games, state):
    lookup = state[
        ["team", "weekly_team_strength", "weekly_strength_rank"]
    ].rename(
        columns={
            "weekly_team_strength": "entering_team_strength",
            "weekly_strength_rank": "entering_strength_rank"
        }
    )

    out = team_games.merge(
        lookup,
        on="team",
        how="left",
        validate="many_to_one"
    )

    opp_lookup = lookup.rename(
        columns={
            "team": "opponent",
            "entering_team_strength": "opponent_entering_strength",
            "entering_strength_rank": "opponent_entering_rank"
        }
    )

    out = out.merge(
        opp_lookup,
        on="opponent",
        how="left",
        validate="many_to_one"
    )

    required = [
        "entering_team_strength",
        "entering_strength_rank",
        "opponent_entering_strength",
        "opponent_entering_rank"
    ]

    if out[required].isna().any().any():
        raise ValueError("Missing entering team/opponent state.")

    return out


def add_weekly_evidence(team_games):
    out = team_games.copy()

    out["weekly_strength_update"] = (
        MAX_WEEKLY_UPDATE
        * np.tanh(out["performance_residual"] / RESIDUAL_SCALE)
    )

    league_points = out["points_for"].mean()

    out["raw_offense_performance"] = out["points_for"] - league_points
    out["raw_defense_performance"] = league_points - out["points_against"]

    out["adjusted_offense_performance"] = (
        out["raw_offense_performance"]
        + OPPONENT_CONTEXT_RATE * out["opponent_entering_strength"]
    )

    out["adjusted_defense_performance"] = (
        out["raw_defense_performance"]
        + OPPONENT_CONTEXT_RATE * out["opponent_entering_strength"]
    )

    out["weekly_offense_adjustment"] = (
        out["adjusted_offense_performance"]
        * OFFENSE_DEFENSE_UPDATE_RATE
    )

    out["weekly_defense_adjustment"] = (
        out["adjusted_defense_performance"]
        * OFFENSE_DEFENSE_UPDATE_RATE
    )

    return out


def build_next_state(prior_state, evidence, target_week):
    # Only persistent state is allowed to carry forward.
    persistent_columns = [
        "team",
        "weekly_strength_rank",
        "weekly_team_strength",
        "cumulative_offense_adjustment",
        "cumulative_defense_adjustment",
        "preseason_strength_rank",
        "preseason_team_strength",
        "preseason_baseline_team_strength",
        "preseason_personnel_adjustment",
        "preseason_personnel_strength",
        "preseason_roster_continuity",
        "preseason_roster_continuity_adjustment"
    ]

    missing = [c for c in persistent_columns if c not in prior_state.columns]
    if missing:
        raise KeyError(f"Prior state is missing persistent columns: {missing}")

    base = prior_state[persistent_columns].copy().rename(
        columns={
            "weekly_strength_rank": "prior_week_strength_rank",
            "weekly_team_strength": "prior_week_team_strength",
            "cumulative_offense_adjustment": "prior_cumulative_offense_adjustment",
            "cumulative_defense_adjustment": "prior_cumulative_defense_adjustment"
        }
    )

    evidence_columns = [
        "team",
        "opponent",
        "opponent_entering_rank",
        "opponent_entering_strength",
        "point_margin",
        "expected_margin",
        "performance_residual",
        "weekly_strength_update",
        "weekly_offense_adjustment",
        "weekly_defense_adjustment"
    ]

    new_evidence = evidence[evidence_columns].copy().rename(
        columns={
            "opponent": "update_opponent",
            "opponent_entering_rank": "update_opponent_rank",
            "opponent_entering_strength": "update_opponent_strength",
            "point_margin": "actual_margin"
        }
    )

    if new_evidence["team"].duplicated().any():
        duplicates = new_evidence.loc[
            new_evidence["team"].duplicated(keep=False),
            "team"
        ].tolist()
        raise ValueError(
            f"Expected one update game per team for this transition. Duplicates: {duplicates}"
        )

    next_state = base.merge(
        new_evidence,
        on="team",
        how="left",
        validate="one_to_one"
    )

    update_cols = [
        "weekly_strength_update",
        "weekly_offense_adjustment",
        "weekly_defense_adjustment"
    ]

    next_state[update_cols] = next_state[update_cols].fillna(0.0)

    next_state["team_strength_change"] = next_state["weekly_strength_update"]

    next_state["weekly_team_strength"] = (
        next_state["prior_week_team_strength"]
        + next_state["team_strength_change"]
    )

    next_state["cumulative_offense_adjustment"] = (
        next_state["prior_cumulative_offense_adjustment"]
        + next_state["weekly_offense_adjustment"]
    )

    next_state["cumulative_defense_adjustment"] = (
        next_state["prior_cumulative_defense_adjustment"]
        + next_state["weekly_defense_adjustment"]
    )

    next_state = next_state.sort_values(
        ["weekly_team_strength", "prior_week_team_strength"],
        ascending=[False, False]
    ).reset_index(drop=True)

    next_state["weekly_strength_rank"] = np.arange(1, len(next_state) + 1)

    next_state["rank_change"] = (
        next_state["prior_week_strength_rank"]
        - next_state["weekly_strength_rank"]
    )

    next_state["state_week"] = target_week

    return next_state


## 5. Create frozen preseason entering state

In [5]:
preseason_state = make_preseason_state(preseason_raw)

assert len(preseason_state) == 32
assert preseason_state["team"].nunique() == 32

display(
    preseason_state[
        ["weekly_strength_rank", "team", "weekly_team_strength"]
    ].sort_values("weekly_strength_rank")
)


,weekly_strength_rank,team,weekly_team_strength
0,1,BUF,4.816514
1,2,LA,4.762452
2,3,SEA,4.363421
3,4,DET,4.018565
4,5,DEN,3.569507
5,6,BAL,3.074111
6,7,HOU,2.810133
7,8,GB,2.536317
8,9,PHI,2.451919
9,10,SF,2.366788


# 6. Week 1 evidence → corrected Week 2 state

In [6]:
week1_results = schedule[
    (schedule["week"] == 1)
    & schedule["home_score"].notna()
    & schedule["away_score"].notna()
].copy()

week1_frozen = preseason_predictions[
    preseason_predictions["week"] == 1
][
    [
        "game_id",
        "expected_home_margin",
        "home_field_adjustment",
        "rest_adjustment"
    ]
].copy()

week1_games = week1_results.merge(
    week1_frozen,
    on="game_id",
    how="left",
    validate="one_to_one"
)

if week1_games["expected_home_margin"].isna().any():
    raise ValueError("Missing frozen Week 1 expected margins.")

print("Completed Week 1 games:", len(week1_games))


Completed Week 1 games: 16


In [7]:
week1_evidence = games_to_team_rows(
    week1_games,
    "expected_home_margin"
)

week1_evidence = attach_entering_state(
    week1_evidence,
    preseason_state
)

week1_evidence = add_weekly_evidence(
    week1_evidence
)

display(
    week1_evidence[
        [
            "team",
            "opponent",
            "entering_strength_rank",
            "opponent_entering_rank",
            "point_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update"
        ]
    ].sort_values("weekly_strength_update", ascending=False)
)


,team,opponent,entering_strength_rank,opponent_entering_rank,point_margin,expected_margin,performance_residual,weekly_strength_update
17,SF,LA,10,2,20.0,-2.395664,22.395664,1.382433
15,KC,DEN,13,5,21.0,-0.503736,21.503736,1.367164
22,CHI,CAR,19,29,22.0,2.717780,19.282220,1.320518
18,ARI,LAC,27,14,12.0,-6.352224,18.352224,1.296744
9,MIN,GB,18,8,17.0,-0.297749,17.297749,1.266274
20,BAL,IND,6,17,18.0,0.778357,17.221643,1.263920
7,JAX,CLE,12,28,24.0,7.627059,16.372941,1.236166
10,LV,MIA,30,25,14.0,-1.486560,15.486560,1.204063
28,NYJ,TEN,31,32,13.0,-0.949910,13.949910,1.140131
8,NYG,DAL,26,20,8.0,-1.305937,9.305937,0.872234


In [8]:
week2_state = build_next_state(
    preseason_state,
    week1_evidence,
    target_week=2
)

assert len(week2_state) == 32
assert week2_state["team"].nunique() == 32

assert np.allclose(
    week2_state["weekly_team_strength"],
    week2_state["prior_week_team_strength"]
    + week2_state["team_strength_change"]
)

display(
    week2_state[
        [
            "weekly_strength_rank",
            "team",
            "weekly_team_strength",
            "prior_week_strength_rank",
            "prior_week_team_strength",
            "team_strength_change",
            "rank_change",
            "update_opponent",
            "performance_residual"
        ]
    ]
)


,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change,update_opponent,performance_residual
0,1,BUF,5.307457,1,4.816514,0.490943,0,HOU,4.757147
1,2,BAL,4.338031,6,3.074111,1.263920,4,IND,17.221643
2,3,SEA,4.233381,3,4.363421,-0.130040,0,NE,-1.216761
3,4,SF,3.749221,10,2.366788,1.382433,6,LA,22.395664
4,5,DET,3.382361,4,4.018565,-0.636204,-1,NO,-6.338146
5,6,LA,3.380019,2,4.762452,-1.382433,-4,SF,-22.395664
6,7,JAX,2.949302,12,1.713136,1.236166,5,CLE,16.372941
7,8,KC,2.669408,13,1.302244,1.367164,5,DEN,21.503736
8,9,HOU,2.319190,7,2.810133,-0.490943,-2,BUF,-4.757147
9,10,DEN,2.202344,5,3.569507,-1.367164,-5,KC,-21.503736


# 7. Week 2 evidence → corrected Week 3 state

Week 2 expected margins are rebuilt from the corrected Week 2 strength state while retaining the frozen game model's home-field and rest adjustments:

`corrected home strength − corrected away strength + frozen HFA + frozen rest adjustment`


In [9]:
week2_results = schedule[
    (schedule["week"] == 2)
    & schedule["home_score"].notna()
    & schedule["away_score"].notna()
].copy()

week2_frozen = preseason_predictions[
    preseason_predictions["week"] == 2
][
    [
        "game_id",
        "expected_home_margin",
        "home_field_adjustment",
        "rest_adjustment"
    ]
].copy()

week2_games = week2_results.merge(
    week2_frozen,
    on="game_id",
    how="left",
    validate="one_to_one"
)

if week2_games[
    ["home_field_adjustment", "rest_adjustment"]
].isna().any().any():
    raise ValueError("Missing frozen Week 2 game context.")

print("Completed Week 2 games:", len(week2_games))


Completed Week 2 games: 16


In [10]:
strength_lookup = week2_state[
    ["team", "weekly_team_strength", "weekly_strength_rank"]
].copy()

week2_games = week2_games.merge(
    strength_lookup.rename(
        columns={
            "team": "home_team",
            "weekly_team_strength": "corrected_home_strength",
            "weekly_strength_rank": "corrected_home_rank"
        }
    ),
    on="home_team",
    how="left",
    validate="many_to_one"
)

week2_games = week2_games.merge(
    strength_lookup.rename(
        columns={
            "team": "away_team",
            "weekly_team_strength": "corrected_away_strength",
            "weekly_strength_rank": "corrected_away_rank"
        }
    ),
    on="away_team",
    how="left",
    validate="many_to_one"
)

week2_games["corrected_expected_home_margin"] = (
    week2_games["corrected_home_strength"]
    - week2_games["corrected_away_strength"]
    + week2_games["home_field_adjustment"]
    + week2_games["rest_adjustment"]
)

week2_games["margin_change_from_frozen_preseason"] = (
    week2_games["corrected_expected_home_margin"]
    - week2_games["expected_home_margin"]
)

display(
    week2_games[
        [
            "away_team",
            "home_team",
            "corrected_away_rank",
            "corrected_home_rank",
            "expected_home_margin",
            "corrected_expected_home_margin",
            "margin_change_from_frozen_preseason"
        ]
    ]
)


,away_team,home_team,corrected_away_rank,corrected_home_rank,expected_home_margin,corrected_expected_home_margin,margin_change_from_frozen_preseason
0,DET,BUF,5,1,2.561476,3.688624,1.127147
1,CAR,ATL,31,24,4.965497,5.932641,0.967143
2,CIN,HOU,18,9,5.313095,4.254344,-1.058751
3,CLE,TB,30,17,6.514949,7.183307,0.668358
4,GB,NYJ,14,29,-6.706516,-4.300111,2.406405
5,IND,KC,20,8,2.352551,4.983635,2.631084
6,JAX,DEN,7,10,3.438904,0.835575,-2.603330
7,LV,LAC,28,19,8.367551,5.866743,-2.500808
8,MIA,SF,27,4,7.116554,9.703051,2.586496
9,MIN,CHI,13,15,1.236553,1.290798,0.054244


In [11]:
week2_evidence = games_to_team_rows(
    week2_games,
    "corrected_expected_home_margin"
)

week2_evidence = attach_entering_state(
    week2_evidence,
    week2_state
)

week2_evidence = add_weekly_evidence(
    week2_evidence
)

display(
    week2_evidence[
        [
            "team",
            "opponent",
            "entering_strength_rank",
            "entering_team_strength",
            "opponent_entering_rank",
            "opponent_entering_strength",
            "point_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update"
        ]
    ].sort_values("weekly_strength_update", ascending=False)
)


,team,opponent,entering_strength_rank,entering_team_strength,opponent_entering_rank,opponent_entering_strength,point_margin,expected_margin,performance_residual,weekly_strength_update
17,CAR,ATL,31,-5.853760,24,-1.684647,31.0,-5.932641,36.932641,1.484741
29,SEA,ARI,3,4.233381,25,-2.381271,24.0,5.575104,18.424896,1.298703
18,CIN,HOU,18,-0.171627,9,2.319190,14.0,-4.254344,18.254344,1.294079
23,LV,LAC,28,-4.489279,19,-0.386063,12.0,-5.866743,17.866743,1.283202
14,DAL,WAS,22,-1.258630,23,-1.563203,17.0,2.068100,14.931900,1.182245
26,NO,BAL,21,-0.919849,2,4.338031,7.0,-7.021408,14.021408,1.143353
15,LA,NYG,6,3.380019,26,-2.583628,22.0,8.270158,13.729842,1.130055
12,NE,PIT,11,2.040228,16,0.897277,17.0,3.630457,13.369543,1.113035
8,SF,MIA,4,3.749221,27,-3.647318,22.0,9.703051,12.296949,1.058390
19,CLE,TB,30,-5.386561,17,0.033219,4.0,-7.183307,11.183307,0.995054


In [12]:
week3_state = build_next_state(
    week2_state,
    week2_evidence,
    target_week=3
)

assert len(week3_state) == 32
assert week3_state["team"].nunique() == 32

assert np.allclose(
    week3_state["weekly_team_strength"],
    week3_state["prior_week_team_strength"]
    + week3_state["team_strength_change"]
)

print("Corrected Week 3 state built successfully.")


Corrected Week 3 state built successfully.


## 8. Final Week 2 → Week 3 review table

In [13]:
final_review = week3_state[
    [
        "weekly_strength_rank",
        "team",
        "weekly_team_strength",
        "prior_week_strength_rank",
        "prior_week_team_strength",
        "team_strength_change",
        "rank_change",
        "update_opponent",
        "update_opponent_rank",
        "actual_margin",
        "expected_margin",
        "performance_residual"
    ]
].copy()

display(final_review)


,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change,update_opponent,update_opponent_rank,actual_margin,expected_margin,performance_residual
0,1,BUF,5.941307,1,5.307457,0.633850,0,DET,5,10.0,3.688624,6.311376
1,2,SEA,5.532084,3,4.233381,1.298703,1,ARI,25,24.0,5.575104,18.424896
2,3,SF,4.807611,4,3.749221,1.058390,1,MIA,27,22.0,9.703051,12.296949
3,4,LA,4.510074,6,3.380019,1.130055,2,NYG,26,22.0,8.270158,13.729842
4,5,BAL,3.194678,2,4.338031,-1.143353,-3,NO,21,-7.0,7.021408,-14.021408
5,6,NE,3.153263,11,2.040228,1.113035,5,PIT,16,17.0,3.630457,13.369543
6,7,DEN,2.823203,10,2.202344,0.620860,3,JAX,7,7.0,0.835575,6.164425
7,8,DET,2.748511,5,3.382361,-0.633850,-3,BUF,1,-10.0,-3.688624,-6.311376
8,9,MIN,2.458757,13,1.741314,0.717443,4,CHI,15,6.0,-1.290798,7.290798
9,10,KC,2.458286,8,2.669408,-0.211121,-2,IND,20,3.0,4.983635,-1.983635


## 9. Two-week audit trail for selected diagnostic teams

In [14]:
KEY_TEAMS = [
    "BUF", "SEA", "NE", "CAR", "CIN",
    "HOU", "DET", "SF", "PIT"
]

week1_audit = week2_state[
    week2_state["team"].isin(KEY_TEAMS)
][
    [
        "team",
        "prior_week_strength_rank",
        "prior_week_team_strength",
        "update_opponent",
        "performance_residual",
        "team_strength_change",
        "weekly_strength_rank",
        "weekly_team_strength"
    ]
].copy()

week1_audit["transition"] = "Preseason → Week 2"

week2_audit = week3_state[
    week3_state["team"].isin(KEY_TEAMS)
][
    [
        "team",
        "prior_week_strength_rank",
        "prior_week_team_strength",
        "update_opponent",
        "performance_residual",
        "team_strength_change",
        "weekly_strength_rank",
        "weekly_team_strength"
    ]
].copy()

week2_audit["transition"] = "Week 2 → Week 3"

audit = pd.concat(
    [week1_audit, week2_audit],
    ignore_index=True
).sort_values(["team", "transition"])

display(audit)


,team,prior_week_strength_rank,prior_week_team_strength,update_opponent,performance_residual,team_strength_change,weekly_strength_rank,weekly_team_strength,transition
0,BUF,1,4.816514,HOU,4.757147,0.490943,1,5.307457,Preseason → Week 2
9,BUF,1,5.307457,DET,6.311376,0.633850,1,5.941307,Week 2 → Week 3
8,CAR,29,-4.533242,CHI,-19.282220,-1.320518,31,-5.853760,Preseason → Week 2
17,CAR,31,-5.853760,ATL,36.932641,1.484741,28,-4.369019,Week 2 → Week 3
7,CIN,21,-0.739434,TB,5.576933,0.567807,18,-0.171627,Preseason → Week 2
14,CIN,18,-0.171627,HOU,18.254344,1.294079,14,1.122452,Week 2 → Week 3
3,DET,4,4.018565,NO,-6.338146,-0.636204,5,3.382361,Preseason → Week 2
13,DET,5,3.382361,BUF,-6.311376,-0.633850,8,2.748511,Week 2 → Week 3
4,HOU,7,2.810133,BUF,-4.757147,-0.490943,9,2.319190,Preseason → Week 2
15,HOU,9,2.319190,CIN,-18.254344,-1.294079,15,1.025111,Week 2 → Week 3


## 10. Leakage and consistency checks

In [15]:
used_weeks = sorted(
    set(week1_evidence["week"].tolist())
    | set(week2_evidence["week"].tolist())
)

if any(w >= 3 for w in used_weeks):
    raise ValueError(
        "DATA LEAKAGE: Week 3 or later results entered the Week 3 state."
    )

assert set(week2_state["team"]) == set(preseason_state["team"])
assert set(week3_state["team"]) == set(preseason_state["team"])

# Each weekly transition should contain one game per team in a normal 32-team week.
if week1_evidence["team"].duplicated().any():
    raise ValueError("Duplicate Week 1 team evidence detected.")

if week2_evidence["team"].duplicated().any():
    raise ValueError("Duplicate Week 2 team evidence detected.")

print("All consistency checks passed.")
print("Result weeks used:", used_weeks)


All consistency checks passed.
Result weeks used: [1, 2]


## 11. Save corrected sequential artifacts

These filenames are separate from the old weekly strength files. Nothing from the original preseason or original Week 2 projection pipeline is overwritten.


In [16]:
WEEKLY_DATA_DIR.mkdir(parents=True, exist_ok=True)

week1_evidence.to_parquet(
    WEEK1_EVIDENCE_OUTPUT_PATH,
    index=False
)

week2_evidence.to_parquet(
    WEEK2_EVIDENCE_OUTPUT_PATH,
    index=False
)

week2_state.to_parquet(
    WEEK2_STATE_OUTPUT_PATH,
    index=False
)

week3_state.to_parquet(
    WEEK3_STATE_OUTPUT_PATH,
    index=False
)

print("Saved corrected sequential artifacts:")
print(" ", WEEK1_EVIDENCE_OUTPUT_PATH)
print(" ", WEEK2_EVIDENCE_OUTPUT_PATH)
print(" ", WEEK2_STATE_OUTPUT_PATH)
print(" ", WEEK3_STATE_OUTPUT_PATH)


Saved corrected sequential artifacts:
  ..\..\data\processed\weekly\week_01_sequential_update_features.parquet
  ..\..\data\processed\weekly\week_02_sequential_update_features.parquet
  ..\..\data\processed\weekly\week_02_sequential_team_strength.parquet
  ..\..\data\processed\weekly\week_03_sequential_team_strength.parquet


## 12. Final output

Use this table for review before connecting the corrected Week 3 state to the injury and Week 3 projection pipeline.


In [17]:
display(
    week3_state[
        [
            "weekly_strength_rank",
            "team",
            "weekly_team_strength",
            "prior_week_strength_rank",
            "prior_week_team_strength",
            "team_strength_change",
            "rank_change"
        ]
    ]
)


,weekly_strength_rank,team,weekly_team_strength,prior_week_strength_rank,prior_week_team_strength,team_strength_change,rank_change
0,1,BUF,5.941307,1,5.307457,0.633850,0
1,2,SEA,5.532084,3,4.233381,1.298703,1
2,3,SF,4.807611,4,3.749221,1.058390,1
3,4,LA,4.510074,6,3.380019,1.130055,2
4,5,BAL,3.194678,2,4.338031,-1.143353,-3
5,6,NE,3.153263,11,2.040228,1.113035,5
6,7,DEN,2.823203,10,2.202344,0.620860,3
7,8,DET,2.748511,5,3.382361,-0.633850,-3
8,9,MIN,2.458757,13,1.741314,0.717443,4
9,10,KC,2.458286,8,2.669408,-0.211121,-2
